# PyTorch twin — same network, a framework instead of hand-rolled math

Same architecture `[784, 64, 32, 10]`, same data split, same optimizer settings,
so the comparison against `01_train_from_scratch.ipynb` is apples-to-apples.

The point: **autograd replaces the `backward()` you wrote by hand.**

In [ ]:
import sys
sys.path.append('..')

import json
import time

import numpy as np
import torch
import torch.nn as nn

from src.metrics import confusion_matrix, precision_recall
from src.utils import CLASS_NAMES, load_fashion_mnist, train_val_split

## Load and split the data (identical to notebook 01)

In [ ]:
x_train, x_test, y_train, y_test = load_fashion_mnist()

# Same split, same seed as the from-scratch baseline.
x_train, x_val, y_train, y_val = train_val_split(x_train, y_train)

print(x_train.shape, x_val.shape, x_test.shape)

## Convert to tensors

Torch wants **samples as rows**, `float32` inputs, and **integer** labels
(`CrossEntropyLoss` takes class indices, not one-hot).

In [ ]:
X_train = torch.from_numpy(x_train.T).float()
y_train_t = torch.from_numpy(y_train).long()

X_val = torch.from_numpy(x_val.T).float()
y_val_t = torch.from_numpy(y_val).long()

X_test = torch.from_numpy(x_test.T).float()
y_test_t = torch.from_numpy(y_test).long()

X_train.shape, y_train_t.shape, X_train.dtype, y_train_t.dtype

## The model

In [ ]:
# TODO: build the same architecture with nn.Sequential.
# Layers, in order: Linear(784 -> 64), ReLU, Linear(64 -> 32), ReLU, Linear(32 -> 10).
# The output layer returns RAW LOGITS — no softmax here (the loss adds it).
model = ...

## Loss and optimizer

In [ ]:
# TODO: define the loss and the optimizer.
# - loss: nn.CrossEntropyLoss()  — combines log_softmax + NLL, so it takes logits.
# - optimizer: torch.optim.SGD(model.parameters(), lr=0.1) to match the baseline.
criterion = ...
optimizer = ...

## Training loop

Autograd does the derivatives — you never write `backward()` again.

Each mini-batch has **five steps**: clear the old gradients, forward pass,
compute the loss, backpropagate, update the weights.

Mirror the baseline: batch size 64, validate each epoch on `X_val`, and stop
early when the validation cost stops improving (patience 3).

In [ ]:
# TODO: implement the training loop.
# - iterate epochs, and inside each epoch iterate mini-batches of the training data
# - five steps per batch: zero_grad -> forward -> loss -> backward -> step
# - after each epoch compute the validation cost WITHOUT tracking gradients
#   (see torch.no_grad()) and append both costs to lists
# - early stopping: stop when the validation cost has not improved for 3 epochs
# - time the whole thing into `train_seconds`, and record `epochs_run`
train_seconds = ...
epochs_run = ...

## Evaluate

In [ ]:
# TODO: predict on the test set and compute accuracy.
# - get logits under torch.no_grad(), then take the argmax over the class axis
# - convert predictions back to numpy for your own metrics
preds = ...
accuracy = ...

In [ ]:
# Reuse the metrics you wrote from scratch — same code as notebook 01.
cm = confusion_matrix(y_test, preds, n_classes=10)
precision, recall = precision_recall(y_test, preds, n_classes=10)

print(f"accuracy {accuracy:.4f} | epochs {epochs_run} | {train_seconds:.1f}s")

## Compare against the from-scratch baseline

In [ ]:
# TODO: save this run to ../results/pytorch.json (mirror the baseline's keys),
# then load ../results/from_scratch.json and print a side-by-side table of
# accuracy, epochs run, and training seconds.

In [ ]:
# TODO (optional): which classes changed most between the two implementations?
# Compare the per-class precision and recall arrays side by side.